# Unit 1: OpenBCI Cyton 设备连接与基础命令

## 学习目标
- 理解 OpenBCI Cyton 的通信架构与串口协议
- 掌握设备初始化与串口连接方法
- 学会发送基础 ASCII 命令并解析设备响应
- 了解通道控制、测试信号、采样率设置等操作

## 核心概念

### 通信架构
OpenBCI Cyton 使用 **RFduino BLE** 进行无线通信：
- **板载 RFduino**：配置为 DEVICE 模式
- **USB Dongle**：配置为 HOST 模式，通过 FTDI 芯片转换为虚拟串口
- **串口参数**：115200 baud, 8-N-1 (8 data bits, No parity, 1 stop bit)

### 启动流程
设备上电后发送 ASCII 启动信息，以 `$$$` 结尾表示就绪：
```text
OpenBCI V3 8-16 channel
ADS1299 Device ID: 0x3E
LIS3DH Device ID: 0x33
Firmware: v2.0.0
$$$
```

### 命令协议要点
- 固件 v2.0.0+：无需命令间延迟，可连续发送
- 固件 v1.x：需要在命令前后添加延时
- 所有命令为单字节 ASCII 字符

## 1. 环境准备与模块导入

In [1]:
import serial
import serial.tools.list_ports
import time
import re
from typing import Optional, Dict, List

print("Modules imported successfully.")

Modules imported successfully.


## 2. 串口检测与端口识别

首先需要找到 Cyton USB Dongle 对应的串口。在 Windows 上是 COM 端口，在 macOS/Linux 上是 `/dev/tty.*` 或 `/dev/ttyUSB*`。

In [2]:
def list_available_ports():
    """
    列出系统中所有可用的串口设备
    
    Returns:
        list: 包含串口信息的字典列表，每个字典包含 device, description, hwid
    """
    ports = serial.tools.list_ports.comports()
    port_list = []
    print("Available Serial Ports:")
    print("-" * 50)
    for port in ports:
        port_info = {
            'device': port.device,
            'description': port.description,
            'hwid': port.hwid
        }
        port_list.append(port_info)
        print(f"  Device: {port.device}")
        print(f"  Description: {port.description}")
        print(f"  Hardware ID: {port.hwid}")
        print("-" * 50)
    return port_list

# 执行端口扫描
available_ports = list_available_ports()

Available Serial Ports:
--------------------------------------------------
  Device: COM4
  Description: USB 串行设备 (COM4)
  Hardware ID: USB VID:PID=0483:5741 SER=6&1C81204A&0&1 LOCATION=1-2.1:x.0
--------------------------------------------------


In [3]:
def find_cyton_port():
    """
    自动查找 OpenBCI Cyton USB Dongle 的串口
    
    FTDI 芯片的 USB Dongle 通常在描述中包含 'FTDI' 或 'USB Serial' 关键字
    
    Returns:
        str or None: 找到的 Cyton 端口号，未找到则返回 None
    """
    ports = serial.tools.list_ports.comports()
    for port in ports:
        # FTDI 芯片特征关键词
        if any(keyword in port.description.upper() for keyword in ['USB', 'FTDI', 'USB SERIAL', 'FT232']):
            print(f"Cyton Dongle found at: {port.device}")
            return port.device
    
    print("No Cyton Dongle detected automatically.")
    print("Please manually specify the COM port.")
    return None

# 自动查找端口
cyton_port = find_cyton_port()
print(f"Detected port: {cyton_port}")

Cyton Dongle found at: COM4
Detected port: COM4


## 3. Cyton 设备连接类

封装串口通信逻辑，创建易于使用的设备控制类。参考 [Cyton SDK 文档](https://docs.openbci.com/Cyton/CytonSDK/)

In [ ]:
class CytonDevice:
    """
    OpenBCI Cyton 设备基础控制类
    
    参数:
        port: 串口设备路径，如 'COM3' (Windows) 或 '/dev/ttyUSB0' (Linux)
        baudrate: 串口波特率，默认为 115200
        timeout: 串口读取超时时间（秒）
    """
    
    # 字符响应结束标记
    RESPONSE_END = b'$$$'
    # 二进制数据流起始字节
    STREAM_START_BYTE = 0xA0

    
    def __init__(self, port: str, baudrate: int = 115200, timeout: int = 2):
        self.port = port                      # 串口设备路径
        self.baudrate = baudrate              # 串口波特率
        self.timeout = timeout                # 串口读取超时时间
        self.ser: Optional[serial.Serial] = None  # 串口对象，初始为空
        self.board_info = ""                  # 设备信息字符串
        self.is_connected = False             # 连接状态标志
        
    def connect(self) -> bool:
        """
        建立与 Cyton 设备的串口连接
        
        串口参数 8-N-1 含义：
        - 8 data bits: 每次传输 8 位数据
        - No parity: 无校验位
        - 1 stop bit: 1 位停止位
        """
        try:
            self.ser = serial.Serial(
                port=self.port,
                baudrate=self.baudrate,
                parity=serial.PARITY_NONE,    # 无校验
                stopbits=serial.STOPBITS_ONE, # 1 位停止位
                bytesize=serial.EIGHTBITS,    # 8 位数据位
                timeout=self.timeout
            )
            print(f"Connected to {self.port} at {self.baudrate} baud.")
            time.sleep(2)  # 等待设备稳定
            
            # 清空缓冲区中的残留数据
            self.ser.reset_input_buffer()
            self.ser.reset_output_buffer()
            
            self.is_connected = True
            return True
            
        except serial.SerialException as e:
            print(f"Connection failed: {e}")
            return False
    
    def disconnect(self):
        """关闭串口连接"""
        if self.ser and self.ser.is_open:
            self.stop_streaming()  # 关闭协议层数据流传输
            self.ser.close()       # 关闭 pyserial 打开的串口句柄
            self.is_connected = False
            print("Disconnected from Cyton device.")
    
    def read_response(self, timeout: float = 2.0) -> str:
        """
        读取设备响应，直到遇到 $$$ 结束标记或超时
        
        Args:
            timeout: 最大等待时间（秒）
        
        Returns:
            str: 设备响应字符串，不包含 $$$ 标记
        """
        if not self.ser or not self.ser.is_open:
            print("Error: Device not connected.")
            return ""
        
        response = b""
        start_time = time.time()
        
        while time.time() - start_time < timeout:
            # 检查串口输入缓冲区是否有数据可读
            if self.ser.in_waiting > 0:
                byte = self.ser.read(1)
                response += byte
                
                # 检查是否收到 $$$ 结束标记
                if self.RESPONSE_END in response:
                    # 先解码为字符串，再移除 $$$
                    return response.decode('utf-8', errors='ignore').replace('$$$', '').strip()
            else:
                time.sleep(0.01)  # 短暂等待
        
        # 超时返回已有内容
        print(f"Timeout after {timeout} seconds. Returning partial response.")
        return response.decode('utf-8', errors='ignore').strip()
    
    def send_command(self, command: str, wait_for_response: bool = True) -> str:
        """
        向设备发送 ASCII 命令
        
        Args:
            command: 单字节 ASCII 命令字符
            wait_for_response: 是否等待设备响应
        
        Returns:
            str: 设备响应内容
        """
        if not self.ser or not self.ser.is_open:
            print("Error: Device not connected.")
            return ""
        
        # 发送命令（需编码为字节）
        self.ser.write(command.encode())
        # 这里 .encode() 没有指定参数 ，在 Python 3 中，它的 默认编码格式是 UTF-8 。
        print(f"Sent command: '{command}'")
        
        if wait_for_response:
            time.sleep(0.1)  # 短延迟确保设备处理完成
            return self.read_response()
        return ""
    
    def reset_board(self) -> str:
        """
        发送 'v' 命令复位设备到默认状态
        这是连接后的首要操作，用于同步设备状态
        """
        response = self.send_command('v')
        time.sleep(2)  # 设备复位需要较长时间
        self.board_info = response
        print(f"Board reset response:\n{response}")
        return response
    
    def get_board_info(self) -> Dict:
        """
        解析设备信息，提取 ADS1299 芯片 ID、加速度计 ID、固件版本等
        """
        info = {
            'board_type': 'Unknown',
            'ads1299_id': 'Unknown',
            'lis3dh_id': 'Unknown',
            'firmware_version': 'Unknown'
        }
        
        # 使用正则表达式提取信息
        # group(1) 就是正则表达式里第一个小括号 () 匹配到的内容
        # group(0) 是整个正则匹配到的完整内容。
        ads_match = re.search(r'ADS1299 Device ID: (0x[0-9A-Fa-f]+)', self.board_info)
        if ads_match:
            info['ads1299_id'] = ads_match.group(1)
        
        lis_match = re.search(r'LIS3DH Device ID: (0x[0-9A-Fa-f]+)', self.board_info)
        if lis_match:
            info['lis3dh_id'] = lis_match.group(1)
        
        fw_match = re.search(r'Firmware: (v[\d.]+)', self.board_info)
        if fw_match:
            info['firmware_version'] = fw_match.group(1)
        
        if '8-16 channel' in self.board_info:
            info['board_type'] = 'Cyton'
        elif '8bit' in self.board_info:
            info['board_type'] = 'Cyton 8-bit (deprecated)'
        
        return info
    
    # ==================== 通道控制命令 ====================
    
    def turn_channel_off(self, channel: int) -> str:
        """
        关闭指定通道（命令: 1-8）
        关闭的通道在数据流中读取值为 0.00
        
        Args:
            channel: 通道号 (1-8)
        """
        if not 1 <= channel <= 8:
            raise ValueError("Channel must be between 1 and 8.")
        command = ['1', '2', '3', '4', '5', '6', '7', '8']
        return self.send_command(command[channel - 1], wait_for_response=False)
    
    def turn_channel_on(self, channel: int) -> str:
        """
        开启指定通道（命令: ! @ # $ % ^ & *）
        对应通道 1-8
        
        Args:
            channel: 通道号 (1-8)
        """
        if not 1 <= channel <= 8:
            raise ValueError("Channel must be between 1 and 8.")
        # ASCII 映射: 1->!, 2->@, 3->#, 4->$, 5->%, 6->^, 7->&, 8->*
        commands = ['!', '@', '#', '$', '%', '^', '&', '*']
        return self.send_command(commands[channel - 1], wait_for_response=False)
    
    def turn_all_channels_off(self):
        """关闭所有 8 个通道"""
        for ch in range(1, 9):
            self.turn_channel_off(ch)
            time.sleep(0.05)
        print("All channels turned OFF.")
    
    def turn_all_channels_on(self):
        """开启所有 8 个通道"""
        for ch in range(1, 9):
            self.turn_channel_on(ch)
            time.sleep(0.05)
        print("All channels turned ON.")
    
    # ==================== 测试信号命令 ====================
    
    def set_test_signal(self, mode: str) -> str:
        """
        设置内部测试信号（用于校准与自检）
        
        模式说明:
            'ground'  (0): 连接内部地 (VDD - VSS)，测量内部噪声
            'slow_1x' (-): 1 倍幅值，慢脉冲
            'fast_1x' (=): 1 倍幅值，快脉冲
            'dc'      (p): 连接直流信号
            'slow_2x' ([): 2 倍幅值，慢脉冲
            'fast_2x' (]): 2 倍幅值，快脉冲
        
        Args:
            mode: 测试信号模式
        """
        mode_map = {
            'ground': '0',
            'slow_1x': '-',
            'fast_1x': '=',
            'dc': 'p',
            'slow_2x': '[',
            'fast_2x': ']'
        }
        if mode not in mode_map:
            raise ValueError(f"Invalid mode. Choose from: {list(mode_map.keys())}")
        return self.send_command(mode_map[mode])
    
    # ==================== 数据流控制 ====================
    
    def start_streaming(self) -> str:
        """
        启动二进制数据流（发送 'b' 命令）
        设备将开始连续发送 33 字节的数据包
        """
        return self.send_command('b', wait_for_response=False)
    
    def stop_streaming(self) -> str:
        """
        停止二进制数据流（发送 's' 命令）
        """
        return self.send_command('s', wait_for_response=False)
    
    # ==================== 采样率设置 ====================
    
    def set_sample_rate(self, rate_code: str) -> str:
        """
        设置采样率（固件 v3.0.0+ 支持）
        
        采样率代码:
            '1': 16000 Hz
            '2': 8000 Hz
            '3': 4000 Hz
            '4': 2000 Hz
            '5': 1000 Hz
            '6': 500 Hz
            '0': 250 Hz (默认)
        
        注意: USB Dongle 模式下最高只能 250 SPS
        高码率需使用 WiFi Shield
        """
        if rate_code not in ['0', '1', '2', '3', '4', '5', '6']:
            raise ValueError("Rate code must be '0'-'6'.")
        # 发送 ~ 进入设置模式，再发送速率代码
        self.send_command('~', wait_for_response=False)
        time.sleep(0.1)
        return self.send_command(rate_code)
    
    def __enter__(self):
        """上下文管理器入口,可以配合 with 使用"""
        self.connect()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """上下文管理器出口，自动断开连接"""
        self.disconnect()

## 4. 实践：连接设备并读取信息

**注意**: 以下代码需要真实的 Cyton 硬件连接才能运行。如果没有硬件，请阅读代码理解逻辑。

In [14]:
# 如果已有硬件连接，请设置正确的端口
# 示例: PORT = 'COM3'  (Windows)
# 示例: PORT = '/dev/ttyUSB0'  (Linux)

PORT = None  # 设置为自动检测或手动指定

if PORT is None:
    PORT = find_cyton_port()

if PORT:
    # 使用上下文管理器连接设备
    with CytonDevice(port=PORT) as cyton:
        # 步骤 1: 复位设备
        print("\n=== Step 1: Reset Board ===")
        cyton.reset_board() # 复位设备、获取设备信息
        
        # 步骤 2: 获取并显示设备信息
        print("\n=== Step 2: Board Information ===")
        info = cyton.get_board_info()
        for key, value in info.items():
            print(f"  {key}: {value}")
        
        # 步骤 3: 开启所有通道
        print("\n=== Step 3: Turn All Channels ON ===")
        cyton.turn_all_channels_on()
        
        # 步骤 4: 设置内部地测试信号（校准用）
        print("\n=== Step 4: Set Test Signal to Internal Ground ===")
        response = cyton.set_test_signal('ground')
        print(f"Response: {response}")
        
        # 步骤 5: 停止测试信号（恢复正常输入）
        print("\n=== Step 5: Set Test Signal to DC ===")
        response = cyton.set_test_signal('dc')
        print(f"Response: {response}")
else:
    print("No Cyton port found. Please connect the device and retry.")
    print("You can still study the code structure without hardware.")

Cyton Dongle found at: COM4
Connected to COM4 at 115200 baud.

=== Step 1: Reset Board ===
Sent command: 'v'
Board reset response:
OpenBCI V3 8-16 channel
On Board ADS1299 Device ID: 0x3e
LIS3DH Device ID: 0x33
Firmware: v3.1.2

=== Step 2: Board Information ===
  board_type: Cyton
  ads1299_id: 0x3e
  lis3dh_id: 0x33
  firmware_version: v3.1.2

=== Step 3: Turn All Channels ON ===
Sent command: '!'
Sent command: '@'
Sent command: '#'
Sent command: '$'
Sent command: '%'
Sent command: '^'
Sent command: '&'
Sent command: '*'
All channels turned ON.

=== Step 4: Set Test Signal to Internal Ground ===
Sent command: '0'
Response: Success: Configured internal test signal.

=== Step 5: Set Test Signal to DC ===
Sent command: 'p'
Response: Success: Configured internal test signal.
Sent command: 's'
Disconnected from Cyton device.


## 5. 命令速查表

| 功能 | 命令 | 说明 |
|------|------|------|
| 复位设备 | `v` | 恢复默认设置 |
| 关闭通道 1-8 | `1`-`8` | 指定通道读数为 0 |
| 开启通道 1-8 | `!` `@` `#` `$` `%` `^` `&` `*` | 对应通道 1-8 |
| 内部地 | `0` | 测量内部噪声 |
| 测试信号 1x 慢 | `-` | 1 倍幅值慢脉冲 |
| 测试信号 1x 快 | `=` | 1 倍幅值快脉冲 |
| 直流信号 | `p` | 连接 DC |
| 测试信号 2x 慢 | `[` | 2 倍幅值慢脉冲 |
| 测试信号 2x 快 | `]` | 2 倍幅值快脉冲 |
| 启动数据流 | `b` | 开始发送二进制数据 |
| 停止数据流 | `s` | 停止发送数据 |

## 6. 常见问题

1. **连接失败**：检查 FTDI 驱动是否正确安装，端口号是否正确
2. **无响应**：确认设备已通电（电池 3-6V），RFduino 指示灯是否亮起
3. **乱码**：检查波特率是否为 115200，串口参数 8-N-1 是否正确
4. **$$$ 未出现**：发送 `v` 命令复位设备，等待 2-3 秒

## 7. 练习任务

1. 修改 `CytonDevice` 类，添加 `turn_selected_channels_on(channels: List[int])` 方法
2. 编写一个函数，循环读取 10 次设备响应并统计响应时间
3. 尝试实现通道配置命令（`x` 命令），设置增益和输入类型

---
*参考资料: [OpenBCI Cyton SDK](https://docs.openbci.com/Cyton/CytonSDK/)*